## Step 1: Install and load libraries

In [1]:
# !python -m pip install -r requirements.txt

In [2]:
import pandas as pd
import numpy as np
import patsy
import statsmodels.api as sm

## Step 2: Load data

In [3]:
df = pd.read_csv("../data/eu_company_pay_data.csv")
df

,employee_id,gender,grade,fte,total_pay,seniority,job_family
0,W001,Male,2,1,27098,14,IT
1,W002,Male,3,1,13223,110,Admin
2,W003,Female,2,1,25301,115,HR
3,W004,Female,2,1,79470,79,IT
4,W005,Male,2,1,24632,36,HR
...,...,...,...,...,...,...,...
495,W496,Female,3,1,49189,109,HR
496,W497,Female,1,1,9674,28,IT
497,W498,Male,1,1,10965,79,HR
498,W499,Male,1,1,24827,95,HR


## Step 3: Prepare Variables

In [4]:
# Calculate full-time equivalent total pay
df["total_pay_fte"] = df["total_pay"] / df["fte"]

# Create log-transformed pay (for percentage interpretation)
df["log_pay"] = np.log(df["total_pay_fte"])

## Step 4: Set reference values

In [5]:
# 1. Gender: define Male as base; no need for C(...) if only two levels
df['gender'] = pd.Categorical(df['gender'],
categories=['Male','Female'])
# 2. Compute minority counts to pick ref_grade
counts = df.groupby(['grade','gender']).size().unstack(fill_value=0)
minority = np.minimum(counts['Male'], counts['Female'])
ref_grade = minority.idxmax() # highest minority count

# 3. Reorder Grade so ref_grade comes first
all_grades = list(np.unique(df['grade']))
other_grades = [g for g in all_grades if g != ref_grade]
df['grade'] = pd.Categorical(df['grade'],
categories=[ref_grade] + other_grades)
# 4. Verify references
assert df['gender'].cat.categories[0] == 'Male'
assert df['grade'].cat.categories[0] == ref_grade

C:\Users\mathi\AppData\Local\Temp\ipykernel_21188\1742187789.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = df.groupby(['grade','gender']).size().unstack(fill_value=0)


## Step 5: Define model

In [6]:
# 1. Build response (y) and design matrix (X)
y, X = patsy.dmatrices(
'log_pay ~ gender * grade + seniority + C(job_family)',
df,
return_type='dataframe'
)
# 2. Drop any columns that are all zeros (no data to estimate)
nonzero = X.columns[(X != 0).any(axis=0)]
X_trimmed = X[nonzero]

## Step 6: Define gender-pay-gap function

In [7]:
def calculate_pay_gap(fit):
    # Pull the fitted parameters into a Series
    b = fit.params
    # Main gender effect (reference grade)
    beta_F = b['gender[T.Female]']
    # For each non-reference grade, get the interaction term or NaN if absent
    beta_ints = {
        g: b.get(f'gender[T.Female]:grade[T.{g}]', np.nan)
        for g in df['grade'].cat.categories
        if  g != ref_grade
    }

    # 1. Reference grade gap
    gap_ref = 1 - np.exp(beta_F)
    rows = [(ref_grade, round(100 * gap_ref, 1))]

    # 2. Other grades: include NaN where interaction was absent
    for g, bi in beta_ints.items():
        gap = 1 - np.exp(beta_F + bi) # bi may be np.nan
        rows.append((g, round(100 * gap, 1)))

    # 3. Assemble into a DataFrame  
    df_gaps = pd.DataFrame(rows, columns=['Grade','AdjustedPayGap'])
    
    return df_gaps

## Step 7: Set initial values

In [8]:
target_gap = 5
fit = sm.OLS(y, X_trimmed).fit()
initial_pay_gap = calculate_pay_gap(fit)
residuals = fit.resid

gmin = np.array([-10.] * len(initial_pay_gap))
gmax = np.array([10.] * len(initial_pay_gap))
gamma = gmin
targets = np.sign(initial_pay_gap["AdjustedPayGap"]) * target_gap


## Step 8: Close pay gaps

In [9]:
for j in range(100):
    df["new_log_pay"] = df["log_pay"]
    for i in range(len(initial_pay_gap)):
        gender_in_scope = "Female" if initial_pay_gap.loc[i, "AdjustedPayGap"] > 0 else "Male"
        idx = np.logical_and(df["grade"] == initial_pay_gap.loc[i, "Grade"], df["gender"] == gender_in_scope)
        df.loc[idx, "new_log_pay"] += np.maximum(residuals[idx], gamma[i]) - residuals[idx]

    new_fit = sm.OLS(df["new_log_pay"], X_trimmed).fit()
    new_pay_gap = calculate_pay_gap(new_fit)

    distance = np.sign(initial_pay_gap["AdjustedPayGap"]) * (new_pay_gap["AdjustedPayGap"] - targets)

    overshot = np.array(distance < 0)
    undershot = np.array(distance > 0)
    gmax[overshot] = gamma[overshot]
    gmin[undershot] = gamma[undershot]

    gamma = (gmax + gmin) / 2

## Step 9: Evaluate Results

In [10]:
df["raise"] = np.round(np.exp(df["new_log_pay"]) - df["total_pay_fte"], 2)
df["new_total_pay_fte"] = df["total_pay_fte"] + df["raise"]

# Cost to close
print(f"Total cost: {sum(df["raise"] * df["fte"])}")

# Number of raises
print(f"Number of raises: {sum(df["raise"]>0)}")

# New pay gap
new_pay_gap


Total cost: 36840.35
Number of raises: 23


,Grade,AdjustedPayGap
0,2,-4.5
1,1,-5.0
2,3,5.0
3,4,5.0
4,5,5.0
